# 03 — DCGAN sur CelebA

Même architecture que le notebook 02, mais appliquée à **CelebA (visages 64×64)**.

Les différences par rapport à CIFAR-10 :
- Résolution **64×64** au lieu de 32×32 → une couche supplémentaire dans G et D
- Dataset CelebA téléchargé via **kagglehub**
- Images plus grandes → temps d'entraînement ~2x plus long

## 1 — GPU

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

if device.type == 'cpu':
    print('ATTENTION : GPU non detecte. Va dans Execution -> Modifier le type execution -> GPU')
else:
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'Memoire : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go')

## 2 — Imports et hyperparamètres

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import os

# Hyperparametres
Z_DIM        = 100
IMG_CHANNELS = 3
IMG_SIZE     = 64     # CelebA en 64x64
G_FEATURES   = 64
D_FEATURES   = 64
BATCH_SIZE   = 128
NUM_EPOCHS   = 100
LR           = 0.0002
BETA1        = 0.5
SAVE_EVERY   = 10

# Dossiers
OUTPUT_DIR = '/content/dcgan_celeba'
IMGS_DIR   = os.path.join(OUTPUT_DIR, 'progression')
os.makedirs(IMGS_DIR, exist_ok=True)

print('Configuration OK')
print(f'  IMG_SIZE   : {IMG_SIZE}x{IMG_SIZE}  <- 64x64 au lieu de 32x32')
print(f'  BATCH_SIZE : {BATCH_SIZE}')
print(f'  NUM_EPOCHS : {NUM_EPOCHS}')

## 3 — Téléchargement de CelebA via kagglehub

kagglehub télécharge CelebA directement sur le serveur Colab.  
Pas besoin de Google Drive ni d'upload manuel.

In [ ]:
!pip install kagglehub -q

import kagglehub
import os

path = kagglehub.dataset_download("jessicali9530/celeba-dataset")
print(f"Dataset téléchargé : {path}")
print(f"Contenu : {os.listdir(path)}")

In [ ]:
# Détection automatique du chemin des images
# La structure peut varier : img_align_celeba/ ou img_align_celeba/img_align_celeba/

CELEBA_ROOT = path
ATTR_CSV    = os.path.join(CELEBA_ROOT, 'list_attr_celeba.csv')
PART_CSV    = os.path.join(CELEBA_ROOT, 'list_eval_partition.csv')

# Vérifie si les images sont dans un sous-dossier ou directement
imgs_path = os.path.join(CELEBA_ROOT, 'img_align_celeba')
if os.path.isdir(os.path.join(imgs_path, 'img_align_celeba')):
    CELEBA_IMGS = os.path.join(imgs_path, 'img_align_celeba')
else:
    CELEBA_IMGS = imgs_path

# Vérification
print(f'Images    : {CELEBA_IMGS}')
print(f'Attrs CSV : {ATTR_CSV}')
print(f'Parts CSV : {PART_CSV}')
print(f'Existe    : imgs={os.path.exists(CELEBA_IMGS)} attrs={os.path.exists(ATTR_CSV)} parts={os.path.exists(PART_CSV)}')
print(f'Nb images : {len(os.listdir(CELEBA_IMGS))}')

## 4 — Dataset CelebA personnalisé

Même classe que dans le notebook 01, adaptée pour Colab.

**Transformation appliquée :**
```
218x178 -> CenterCrop(140) -> 140x140 -> Resize(64) -> 64x64 -> Normalize[-1,1]
```
- `CenterCrop(140)` : garde uniquement le visage, enlève le cou et le fond
- `Resize(64)` : résolution cible pour cette architecture

In [ ]:
class CelebADataset(Dataset):
    """
    Dataset CelebA personnalise.
    Pour un GAN on n'a pas besoin des attributs — juste les images.
    """
    def __init__(self, root_imgs, partition_csv, split='train', transform=None):
        self.root_imgs = root_imgs
        self.transform = transform

        split_map = {'train': 0, 'valid': 1, 'test': 2}
        partitions = pd.read_csv(partition_csv)
        train_imgs = partitions[partitions['partition'] == split_map[split]]
        self.filenames = train_imgs['image_id'].tolist()

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.root_imgs, self.filenames[idx])).convert('RGB')
        if self.transform:
            img = self.transform(img)
        # On retourne (image, 0) pour garder la meme interface que les datasets torchvision
        # Le 0 est un label factice — le GAN n'en a pas besoin
        return img, 0


celeba_transform = transforms.Compose([
    transforms.CenterCrop(140),
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset = CelebADataset(
    root_imgs     = CELEBA_IMGS,
    partition_csv = PART_CSV,
    split         = 'train',
    transform     = celeba_transform
)

dataloader = DataLoader(
    dataset,
    batch_size = BATCH_SIZE,
    shuffle    = True,
    num_workers= 2,
    pin_memory = True
)

print(f'Dataset CelebA  : {len(dataset)} images (split train)')
print(f'Batches/epoch   : {len(dataloader)}')

# Verification
imgs, _ = next(iter(dataloader))
print(f'Shape batch     : {imgs.shape}  -> (batch, 3, 64, 64)')
print(f'Valeurs         : min={imgs.min():.2f}  max={imgs.max():.2f}')

In [ ]:
# Visualisation rapide des vraies images pour verifier le preprocessing
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('CelebA — Vraies images apres preprocessing (64x64)', fontsize=11)

for i, ax in enumerate(axes.flat):
    img = (imgs[i] * 0.5 + 0.5).clamp(0,1).permute(1,2,0).numpy()
    ax.imshow(img)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 5 — Générateur 64×64

Par rapport au DCGAN CIFAR-10 (32×32), on ajoute **une couche supplémentaire** pour atteindre 64×64 :

```
z(100,1,1) -> (512,4,4) -> (256,8,8) -> (128,16,16) -> (64,32,32) -> (3,64,64)
                                                         ^nouvelle couche^
```

In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim, features, channels):
        super().__init__()
        self.net = nn.Sequential(
            # z(100,1,1) -> (512, 4, 4)
            nn.ConvTranspose2d(z_dim, features * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(features * 8),
            nn.ReLU(True),
            # (512, 4, 4) -> (256, 8, 8)
            nn.ConvTranspose2d(features * 8, features * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features * 4),
            nn.ReLU(True),
            # (256, 8, 8) -> (128, 16, 16)
            nn.ConvTranspose2d(features * 4, features * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features * 2),
            nn.ReLU(True),
            # (128, 16, 16) -> (64, 32, 32)  <- couche supplementaire vs CIFAR-10
            nn.ConvTranspose2d(features * 2, features, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features),
            nn.ReLU(True),
            # (64, 32, 32) -> (3, 64, 64)
            nn.ConvTranspose2d(features, channels, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)


G = Generator(Z_DIM, G_FEATURES, IMG_CHANNELS).to(device)
z_test = torch.randn(4, Z_DIM, 1, 1).to(device)
print(f'Sortie G : {G(z_test).shape}  -> doit etre (4, 3, 64, 64)')
print(f'Parametres G : {sum(p.numel() for p in G.parameters()):,}')

## 6 — Discriminateur 64×64

Idem — une couche supplémentaire pour traiter le 64×64 :

```
(3,64,64) -> (64,32,32) -> (128,16,16) -> (256,8,8) -> (512,4,4) -> score
              ^nouvelle couche^
```

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, channels, features):
        super().__init__()
        self.net = nn.Sequential(
            # (3,64,64) -> (64,32,32) — pas de BatchNorm sur la 1ere couche
            nn.Conv2d(channels, features, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # (64,32,32) -> (128,16,16)  <- couche supplementaire vs CIFAR-10
            nn.Conv2d(features, features * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # (128,16,16) -> (256,8,8)
            nn.Conv2d(features * 2, features * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # (256,8,8) -> (512,4,4)
            nn.Conv2d(features * 4, features * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # (512,4,4) -> (1,1,1)
            nn.Conv2d(features * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, img):
        return self.net(img).view(-1)


D = Discriminator(IMG_CHANNELS, D_FEATURES).to(device)
img_test = torch.randn(4, IMG_CHANNELS, IMG_SIZE, IMG_SIZE).to(device)
print(f'Sortie D : {D(img_test).shape}  -> doit etre (4,)')
print(f'Parametres D : {sum(p.numel() for p in D.parameters()):,}')

## 7 — Initialisation, perte, optimiseurs

In [ ]:
def init_weights(model):
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.normal_(m.weight.data, 0.0, 0.02)
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.normal_(m.weight.data, 1.0, 0.02)
            nn.init.constant_(m.bias.data, 0)

G.apply(init_weights)
D.apply(init_weights)

criterion   = nn.BCELoss()
optimizer_G = optim.Adam(G.parameters(), lr=LR, betas=(BETA1, 0.999))
optimizer_D = optim.Adam(D.parameters(), lr=LR, betas=(BETA1, 0.999))

z_fixed = torch.randn(64, Z_DIM, 1, 1).to(device)

print('Prets pour l entrainement.')

## 8 — Entraînement

In [ ]:
def save_snapshot(generator, z, epoch):
    generator.eval()
    with torch.no_grad():
        imgs = (generator(z).cpu() * 0.5 + 0.5).clamp(0, 1)
    fig, axes = plt.subplots(8, 8, figsize=(12, 12))
    fig.suptitle(f'Epoch {epoch}', fontsize=12)
    for i, ax in enumerate(axes.flat):
        ax.imshow(imgs[i].permute(1, 2, 0).numpy())
        ax.axis('off')
    plt.tight_layout()
    path = os.path.join(IMGS_DIR, f'epoch_{epoch:04d}.png')
    plt.savefig(path, dpi=80, bbox_inches='tight')
    plt.close()
    generator.train()
    return path


losses_G        = []
losses_D        = []
saved_snapshots = []

print(f'Debut entrainement — {NUM_EPOCHS} epochs | {len(dataloader)} batches/epoch')
print('-' * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    sum_G = 0
    sum_D = 0

    for real_imgs, _ in dataloader:
        real_imgs = real_imgs.to(device)
        n         = real_imgs.size(0)
        real_lbl  = torch.ones(n,  device=device)
        fake_lbl  = torch.zeros(n, device=device)

        # Etape 1 : entrainer D
        optimizer_D.zero_grad()
        loss_real = criterion(D(real_imgs), real_lbl)
        z         = torch.randn(n, Z_DIM, 1, 1, device=device)
        fake_imgs = G(z)
        loss_fake = criterion(D(fake_imgs.detach()), fake_lbl)
        loss_D    = loss_real + loss_fake
        loss_D.backward()
        optimizer_D.step()

        # Etape 2 : entrainer G
        optimizer_G.zero_grad()
        loss_G = criterion(D(fake_imgs), real_lbl)
        loss_G.backward()
        optimizer_G.step()

        sum_G += loss_G.item()
        sum_D += loss_D.item()

    avg_G = sum_G / len(dataloader)
    avg_D = sum_D / len(dataloader)
    losses_G.append(avg_G)
    losses_D.append(avg_D)

    if epoch % SAVE_EVERY == 0 or epoch == 1:
        path = save_snapshot(G, z_fixed, epoch)
        saved_snapshots.append((epoch, path))
        print(f'Epoch [{epoch:3d}/{NUM_EPOCHS}] G={avg_G:.4f} D={avg_D:.4f} | snapshot')
    elif epoch % 5 == 0:
        print(f'Epoch [{epoch:3d}/{NUM_EPOCHS}] G={avg_G:.4f} D={avg_D:.4f}')

print('-' * 65)
print('Termine.')

## 9 — Visualisation

In [ ]:
# Courbes de perte
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(losses_G, label='Generateur',     color='royalblue')
ax.plot(losses_D, label='Discriminateur', color='tomato')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Evolution des pertes — DCGAN CelebA')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'losses.png'), dpi=100)
plt.show()

In [ ]:
from matplotlib.image import imread

# Progression complete
n = len(saved_snapshots)
fig, axes = plt.subplots(1, n, figsize=(n * 3, 3))
fig.suptitle('Progression DCGAN CelebA', fontsize=13)

for i, (epoch, path) in enumerate(saved_snapshots):
    ax = axes[i] if n > 1 else axes
    ax.imshow(imread(path))
    ax.set_title(f'Epoch {epoch}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'progression.png'), dpi=100, bbox_inches='tight')
plt.show()

## 10 — FID

In [ ]:
!pip install pytorch-fid -q

from torchvision.utils import save_image

real_dir = '/content/fid_real_celeba'
fake_dir = '/content/fid_fake_celeba'
os.makedirs(real_dir, exist_ok=True)
os.makedirs(fake_dir, exist_ok=True)

# 10 000 vraies images
eval_dataset = CelebADataset(
    root_imgs     = CELEBA_IMGS,
    partition_csv = PART_CSV,
    split         = 'test',
    transform     = celeba_transform
)
for i, (img, _) in enumerate(eval_dataset):
    save_image((img * 0.5 + 0.5).clamp(0,1), f'{real_dir}/{i:05d}.png')
    if i >= 9999: break
print(f'Vraies images : {len(os.listdir(real_dir))}')

# 10 000 fausses images
G.eval()
idx = 0
with torch.no_grad():
    while idx < 10000:
        z    = torch.randn(128, Z_DIM, 1, 1).to(device)
        imgs = (G(z).cpu() * 0.5 + 0.5).clamp(0,1)
        for img in imgs:
            save_image(img, f'{fake_dir}/{idx:05d}.png')
            idx += 1
            if idx >= 10000: break
print(f'Fausses images : {len(os.listdir(fake_dir))}')

# Calcul FID
!python -m pytorch_fid {real_dir} {fake_dir} --device cuda

## 11 — Sauvegarde

In [ ]:
ckpt = {
    'epoch'        : NUM_EPOCHS,
    'G_state_dict' : G.state_dict(),
    'D_state_dict' : D.state_dict(),
    'optimizer_G'  : optimizer_G.state_dict(),
    'optimizer_D'  : optimizer_D.state_dict(),
    'losses_G'     : losses_G,
    'losses_D'     : losses_D,
}
ckpt_path = os.path.join(OUTPUT_DIR, 'dcgan_celeba_final.pt')
torch.save(ckpt, ckpt_path)
print(f'Checkpoint : {ckpt_path}')

from google.colab import files
files.download(ckpt_path)
files.download(os.path.join(OUTPUT_DIR, 'progression.png'))
files.download(os.path.join(OUTPUT_DIR, 'losses.png'))